In [15]:
from pathlib import Path

import polars as pl

from fedaugment.utils import cd_to_git_root

cd_to_git_root()

_ = pl.Config.set_tbl_cols(-1)
_ = pl.Config.set_tbl_rows(20)
_ = pl.Config.set_tbl_width_chars(200)
_ = pl.Config.set_float_precision(3)

2026-04-05 12:26:55.403 | DEBUG    | fedaugment.utils.misc:cd_to_git_root:154 - Changed working directory to: /home/leonard/Dokumente/GitHub/fedaugment-public


In [16]:
# Load results per dataset and experiment group
base_path = Path("logs/augmentations")

# Dict: dataset -> {model_name: df}
results: dict[str, pl.DataFrame] = {}

for dataset_path in base_path.iterdir():
    if not dataset_path.is_dir():
        continue

    dataset = dataset_path.name
    centralized_path = dataset_path / "centralized"

    df_list = []
    for f in centralized_path.glob("*.csv"):
        model_name = f.stem.split("-")[0]
        df = pl.read_csv(f)
        df_list.append(
            df.select(
                precision_1=df["precision@1"].mean(),
                recall_1=df["recall@1"].mean(),
                precision_5=df["precision@5"].mean(),
                recall_5=df["recall@5"].mean(),
                precision_10=df["precision@10"].mean(),
                recall_10=df["recall@10"].mean(),
                precision_20=df["precision@20"].mean(),
                recall_20=df["recall@20"].mean(),
            ).with_columns(pl.lit(model_name).alias("model"))
        )
    results[dataset] = pl.concat(df_list)

    print(f"{dataset}: {len(results[dataset])}")

omnimatch_culture_test: 13
santos-union: 13
freyja: 13
webtable-union: 9
omnimatch_city_test: 13


In [17]:
results["freyja"]

precision_1,recall_1,precision_5,recall_5,precision_10,recall_10,precision_20,recall_20,model
f64,f64,f64,f64,f64,f64,f64,f64,str
0.960,0.031,0.920,0.151,0.892,0.292,0.853,0.557,"""mpnet"""
0.920,0.030,0.944,0.155,0.934,0.306,0.906,0.594,"""gtr_t5"""
0.940,0.031,0.912,0.149,0.916,0.300,0.863,0.564,"""qwen3_06b"""
0.940,0.031,0.940,0.154,0.940,0.308,0.833,0.543,"""llama_nemotron_8b"""
0.920,0.030,0.896,0.146,0.916,0.299,0.881,0.575,"""gte_base"""
0.900,0.029,0.916,0.150,0.918,0.300,0.840,0.551,"""distilroberta"""
0.940,0.031,0.948,0.155,0.940,0.308,0.889,0.581,"""qwen3_8b"""
0.960,0.031,0.924,0.151,0.922,0.302,0.874,0.571,"""qwen3_4b"""
0.960,0.031,0.948,0.155,0.936,0.306,0.898,0.587,"""mini_l12"""


In [18]:
metric_cols = [col for col in results["freyja"].columns if col != "model"]
rank_rows: list[pl.DataFrame] = []

for dataset, df in results.items():
    if dataset == "webtable-union":
        # webtable-union only uses 8 views, so skip it
        continue
    for metric in metric_cols:
        ranked = (
            df.select(["model", metric])
            .sort(metric, descending=True)
            .with_row_index(name="rank")
            .with_columns(
                pl.lit(dataset).alias("dataset"),
                pl.lit(metric).alias("metric"),
                (pl.col("rank") + 1).alias("rank"),
            )
            .select(["dataset", "model", "metric", "rank"])
        )
        rank_rows.append(ranked)

rank_df = pl.concat(rank_rows)

In [19]:
# Average rank per model across metrics and datasets
avg_rank = (
    rank_df.group_by(["model"]).agg(pl.col("rank").mean().alias("avg_rank")).sort(["avg_rank"])
)

avg_rank

model,avg_rank
str,f64
"""qwen3_8b""",4.375
"""mini_l12""",4.531
"""qwen3_4b""",4.594
"""qwen3_06b""",6.562
"""llama_nemotron_8b""",6.562
"""gte_base""",6.688
"""mini_l6""",6.812
"""mpnet""",7.156
"""gtr_t5""",7.562


In [20]:
# Average rank per model and metric across datasets
avg_rank_per_metric = rank_df.group_by(["model", "metric"]).agg(
    pl.col("rank").mean().alias("avg_rank")
)

avg_rank_per_metric.pivot("metric", index="model", values="avg_rank", sort_columns=True).sort(
    ["model"]
)

model,precision_1,precision_10,precision_20,precision_5,recall_1,recall_10,recall_20,recall_5
str,f64,f64,f64,f64,f64,f64,f64,f64
"""deepjoin_dj_orig""",8.000,9.667,11.667,7.667,8.000,9.667,11.667,7.667
"""distilroberta""",8.250,10.250,11.750,9.500,6.500,10.500,11.750,9.500
"""gte_base""",7.500,6.250,4.750,7.500,8.500,7.250,4.250,7.500
"""gtr_t5""",9.000,7.500,7.750,6.750,6.750,7.250,8.000,7.500
"""kalm_gemma3_12b""",5.250,9.250,10.500,8.500,8.000,9.250,10.000,8.250
"""kalm_instruct_v2_5""",8.500,7.250,7.500,8.750,8.500,6.500,7.000,8.250
"""llama_nemotron_8b""",7.250,4.750,8.750,5.500,6.500,5.250,8.750,5.750
"""mini_l12""",4.250,4.750,3.000,4.750,6.250,4.750,3.500,5.000
"""mini_l6""",4.250,7.750,5.500,8.750,4.750,8.000,6.500,9.000


In [21]:
# Detailed ranks per dataset
for dataset, result_df in results.items():
    print(f"Dataset: {dataset}")
    display(
        result_df.select(
            result_df.sort("precision_1", descending=True)["model"].alias("precision_1"),
            result_df.sort("recall_1", descending=True)["model"].alias("recall_1"),
            result_df.sort("precision_5", descending=True)["model"].alias("precision_5"),
            result_df.sort("recall_5", descending=True)["model"].alias("recall_5"),
            result_df.sort("precision_10", descending=True)["model"].alias("precision_10"),
            result_df.sort("recall_10", descending=True)["model"].alias("recall_10"),
            result_df.sort("precision_20", descending=True)["model"].alias("precision_20"),
            result_df.sort("recall_20", descending=True)["model"].alias("recall_20"),
        )
    )

Dataset: omnimatch_culture_test


precision_1,recall_1,precision_5,recall_5,precision_10,recall_10,precision_20,recall_20
str,str,str,str,str,str,str,str
"""qwen3_4b""","""qwen3_4b""","""qwen3_8b""","""qwen3_4b""","""qwen3_4b""","""qwen3_4b""","""gte_base""","""mpnet"""
"""mini_l12""","""llama_nemotron_8b""","""qwen3_4b""","""qwen3_8b""","""llama_nemotron_8b""","""qwen3_06b""","""qwen3_4b""","""gte_base"""
"""gte_base""","""mini_l6""","""gte_base""","""gte_base""","""qwen3_8b""","""qwen3_8b""","""mpnet""","""qwen3_8b"""
"""mini_l6""","""mini_l12""","""mini_l12""","""mini_l12""","""qwen3_06b""","""llama_nemotron_8b""","""qwen3_8b""","""qwen3_4b"""
"""gtr_t5""","""gte_base""","""llama_nemotron_8b""","""llama_nemotron_8b""","""gte_base""","""gte_base""","""qwen3_06b""","""qwen3_06b"""
"""llama_nemotron_8b""","""gtr_t5""","""deepjoin_dj_orig""","""deepjoin_dj_orig""","""mini_l12""","""mini_l12""","""llama_nemotron_8b""","""llama_nemotron_8b"""
"""qwen3_8b""","""qwen3_8b""","""mpnet""","""qwen3_06b""","""gtr_t5""","""gtr_t5""","""mini_l12""","""mini_l12"""
"""mpnet""","""distilroberta""","""qwen3_06b""","""mpnet""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5"""
"""qwen3_06b""","""qwen3_06b""","""gtr_t5""","""kalm_instruct_v2_5""","""mpnet""","""mpnet""","""mini_l6""","""mini_l6"""


Dataset: santos-union


precision_1,recall_1,precision_5,recall_5,precision_10,recall_10,precision_20,recall_20
str,str,str,str,str,str,str,str
"""kalm_instruct_v2_5""","""mpnet""","""mpnet""","""mpnet""","""llama_nemotron_8b""","""llama_nemotron_8b""","""mini_l12""","""mini_l12"""
"""kalm_gemma3_12b""","""gtr_t5""","""llama_nemotron_8b""","""llama_nemotron_8b""","""gte_base""","""qwen3_4b""","""qwen3_4b""","""qwen3_4b"""
"""mini_l6""","""distilroberta""","""gte_base""","""gte_base""","""qwen3_4b""","""mini_l12""","""mpnet""","""gte_base"""
"""mini_l12""","""qwen3_8b""","""distilroberta""","""distilroberta""","""mini_l12""","""gte_base""","""mini_l6""","""qwen3_8b"""
"""qwen3_4b""","""mini_l6""","""qwen3_4b""","""qwen3_4b""","""qwen3_8b""","""qwen3_8b""","""llama_nemotron_8b""","""llama_nemotron_8b"""
"""qwen3_8b""","""kalm_instruct_v2_5""","""mini_l12""","""mini_l12""","""kalm_gemma3_12b""","""kalm_gemma3_12b""","""gte_base""","""kalm_instruct_v2_5"""
"""distilroberta""","""starmie_webtable""","""qwen3_06b""","""qwen3_8b""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""qwen3_8b""","""mpnet"""
"""gte_base""","""qwen3_06b""","""mini_l6""","""kalm_gemma3_12b""","""mini_l6""","""mini_l6""","""kalm_instruct_v2_5""","""mini_l6"""
"""llama_nemotron_8b""","""llama_nemotron_8b""","""kalm_gemma3_12b""","""qwen3_06b""","""mpnet""","""mpnet""","""qwen3_06b""","""kalm_gemma3_12b"""


Dataset: freyja


precision_1,recall_1,precision_5,recall_5,precision_10,recall_10,precision_20,recall_20
str,str,str,str,str,str,str,str
"""mini_l6""","""mini_l6""","""mini_l12""","""mini_l12""","""mini_l6""","""mini_l6""","""mini_l6""","""mini_l6"""
"""kalm_gemma3_12b""","""kalm_gemma3_12b""","""qwen3_8b""","""qwen3_8b""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""gtr_t5""","""gtr_t5"""
"""mpnet""","""mpnet""","""mini_l6""","""mini_l6""","""llama_nemotron_8b""","""llama_nemotron_8b""","""mini_l12""","""mini_l12"""
"""qwen3_4b""","""qwen3_4b""","""gtr_t5""","""gtr_t5""","""qwen3_8b""","""qwen3_8b""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5"""
"""mini_l12""","""mini_l12""","""llama_nemotron_8b""","""kalm_instruct_v2_5""","""mini_l12""","""gtr_t5""","""qwen3_8b""","""qwen3_8b"""
"""qwen3_06b""","""qwen3_06b""","""kalm_instruct_v2_5""","""llama_nemotron_8b""","""gtr_t5""","""mini_l12""","""gte_base""","""gte_base"""
"""llama_nemotron_8b""","""kalm_instruct_v2_5""","""qwen3_4b""","""qwen3_4b""","""qwen3_4b""","""qwen3_4b""","""qwen3_4b""","""qwen3_4b"""
"""qwen3_8b""","""qwen3_8b""","""kalm_gemma3_12b""","""kalm_gemma3_12b""","""distilroberta""","""distilroberta""","""qwen3_06b""","""qwen3_06b"""
"""kalm_instruct_v2_5""","""llama_nemotron_8b""","""mpnet""","""mpnet""","""gte_base""","""qwen3_06b""","""mpnet""","""mpnet"""


Dataset: webtable-union


precision_1,recall_1,precision_5,recall_5,precision_10,recall_10,precision_20,recall_20
str,str,str,str,str,str,str,str
"""starmie_webtable""","""starmie_webtable""","""starmie_webtable""","""starmie_webtable""","""starmie_webtable""","""starmie_webtable""","""starmie_webtable""","""starmie_webtable"""
"""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5""","""kalm_instruct_v2_5"""
"""gtr_t5""","""qwen3_06b""","""gtr_t5""","""gtr_t5""","""gtr_t5""","""gtr_t5""","""distilroberta""","""distilroberta"""
"""qwen3_06b""","""gtr_t5""","""distilroberta""","""distilroberta""","""distilroberta""","""distilroberta""","""gtr_t5""","""gtr_t5"""
"""distilroberta""","""mpnet""","""qwen3_06b""","""qwen3_06b""","""qwen3_06b""","""qwen3_06b""","""mpnet""","""mpnet"""
"""mpnet""","""distilroberta""","""mpnet""","""mpnet""","""mpnet""","""mpnet""","""qwen3_06b""","""qwen3_06b"""
"""mini_l6""","""mini_l6""","""mini_l6""","""mini_l6""","""mini_l6""","""mini_l6""","""mini_l6""","""mini_l6"""
"""mini_l12""","""mini_l12""","""mini_l12""","""mini_l12""","""mini_l12""","""mini_l12""","""mini_l12""","""mini_l12"""
"""gte_base""","""gte_base""","""gte_base""","""gte_base""","""gte_base""","""gte_base""","""gte_base""","""gte_base"""


Dataset: omnimatch_city_test


precision_1,recall_1,precision_5,recall_5,precision_10,recall_10,precision_20,recall_20
str,str,str,str,str,str,str,str
"""deepjoin_dj_orig""","""deepjoin_dj_orig""","""qwen3_8b""","""qwen3_8b""","""qwen3_8b""","""qwen3_8b""","""mini_l12""","""qwen3_06b"""
"""qwen3_06b""","""distilroberta""","""gtr_t5""","""qwen3_06b""","""qwen3_06b""","""qwen3_06b""","""qwen3_06b""","""qwen3_8b"""
"""distilroberta""","""qwen3_06b""","""qwen3_06b""","""gtr_t5""","""qwen3_4b""","""qwen3_4b""","""qwen3_8b""","""mini_l12"""
"""kalm_gemma3_12b""","""mini_l12""","""deepjoin_dj_orig""","""deepjoin_dj_orig""","""mini_l12""","""mini_l12""","""qwen3_4b""","""qwen3_4b"""
"""qwen3_8b""","""kalm_gemma3_12b""","""kalm_gemma3_12b""","""kalm_gemma3_12b""","""gtr_t5""","""gtr_t5""","""mpnet""","""mpnet"""
"""mini_l12""","""llama_nemotron_8b""","""qwen3_4b""","""qwen3_4b""","""deepjoin_dj_orig""","""deepjoin_dj_orig""","""gte_base""","""gte_base"""
"""llama_nemotron_8b""","""gte_base""","""mpnet""","""mpnet""","""kalm_gemma3_12b""","""kalm_gemma3_12b""","""kalm_gemma3_12b""","""kalm_gemma3_12b"""
"""gte_base""","""gtr_t5""","""mini_l12""","""kalm_instruct_v2_5""","""mpnet""","""mpnet""","""mini_l6""","""mini_l6"""
"""mini_l6""","""qwen3_8b""","""kalm_instruct_v2_5""","""mini_l12""","""gte_base""","""kalm_instruct_v2_5""","""gtr_t5""","""gtr_t5"""
